In [1]:
# Create and display the Shor's algorithm circuit for N=15, a=7 (as an example)
qc = build_shor_circuit_for_15(a=7, t=8)

# Display the circuit with a reasonable width
print("Full Shor's Algorithm Circuit for N=15, a=7")
qc.draw(output='mpl', fold=100)  # fold=100 prevents line wrapping for better visibility

NameError: name 'build_shor_circuit_for_15' is not defined

# Shor's Algorithm: Explanation and Worked Implementation

This notebook walks through Shor's algorithm for integer factorisation. It mixes conceptual notes with a runnable quantum circuit (demonstrated on a small composite number) and classical post-processing to extract the factors. The goal is to be self-contained: you can read the explanation and execute the cells in order to see the algorithm in action.


## How to use this notebook
- Run the cells top to bottom. If you have not installed Qiskit, run the optional install cell below.
- The implementation demonstrates factoring 15 (the smallest non-trivial case), which keeps the circuit depth manageable for simulation. The classical helpers are written generically so you can plug in other small semiprimes if you adapt the modular multiplication gadgets.
- Sections are numbered: overview ➜ math background ➜ quantum subroutines ➜ building the circuit ➜ running the demo ➜ discussion and extensions.


In [ ]:
# Optional: install qiskit if your environment does not have it yet
# Remove the leading '!' if you copy this code into a plain .py file.
# Running this cell is safe to skip if qiskit is already installed.
try:
    import qiskit  # noqa: F401
except ImportError:
    %pip install "qiskit>=1.1" "qiskit-aer>=0.15"



## 1. Conceptual overview
Shor's algorithm factors an odd composite integer \(N\) by reducing the problem to finding the order \(r\) of a random integer \(a\) that is coprime with \(N\). The key observations:

1. If \(\gcd(a, N) > 1\), we already found a non-trivial factor.
2. If the (multiplicative) order \(r\) of \(a\) modulo \(N\) is even and \(a^{r/2} \not\equiv -1 \pmod N\), then
   \[ \gcd(a^{r/2} - 1, N) \] and \[ \gcd(a^{r/2} + 1, N) \] yield non-trivial factors.
3. The quantum subroutine efficiently estimates \(r\) using the Quantum Phase Estimation (QPE) framework applied to modular exponentiation.

We will:
- Build classical helpers (GCD checks, continued fractions for phase estimation, post-processing).
- Implement the modular-exponentiation oracle for a small example (\(N = 15\)).
- Assemble the full QPE-style circuit, simulate it, and extract \(r\).
- Recover the factors of \(N\) from the measured order.

This notebook keeps the quantum circuit modest so it runs comfortably on a laptop simulator. Scaling to large \(N\) requires deeper modular multiplication circuits or access to fault-tolerant hardware.


In [ ]:
import math
import random
import json
import time
from fractions import Fraction
from typing import List, Tuple

# Core Qiskit imports
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit.circuit.library import QFT

import os
from pathlib import Path

# Write debug logs to a workspace-relative file (robust across environments)
DEBUG_LOG_PATH = str(Path(os.getcwd()) / "shors_debug.log")
print("DEBUG_LOG_PATH =", DEBUG_LOG_PATH)

# Ensure the file is creatable (helps catch permission/path issues early)
try:
    Path(DEBUG_LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
    Path(DEBUG_LOG_PATH).touch(exist_ok=True)
except Exception as _e:
    print("WARNING: could not create debug log file:", repr(_e))

# Robust simulator selection across Qiskit versions
try:
    # Preferred: dedicated Aer simulator class (qiskit-aer installed)
    from qiskit_aer import AerSimulator as _AerSimulatorClass

    def get_simulator():
        return _AerSimulatorClass()

except ImportError:
    # Fallback: use legacy Aer provider bundled with qiskit
    from qiskit import Aer as _AerProvider

    def get_simulator():
        # Name is 'qasm_simulator' in the legacy Aer provider
        return _AerProvider.get_backend("qasm_simulator")

# Utility RNG for reproducibility in the demo
random.seed(42)



DEBUG_LOG_PATH = c:\Users\chpsu\Documents\project Q codes\shors algo\shors_debug.log


## 2. Classical ingredients
Before building the quantum piece, we need a few classical utilities:
- Greatest common divisor (for the quick win case \(\gcd(a, N) > 1\)).
- Continued fraction expansion to turn measured phases into good rational approximations.
- Order-to-factor post-processing: given \(r\), derive non-trivial factors of \(N\) if possible.


In [ ]:
def continued_fraction_expansion(x: float, max_depth: int = 20) -> List[int]:
    """Return the continued fraction coefficients of x."""
    coeffs = []
    value = x
    for _ in range(max_depth):
        integer_part = math.floor(value)
        coeffs.append(integer_part)
        fractional_part = value - integer_part
        if fractional_part == 0:
            break
        value = 1 / fractional_part
    return coeffs


def convergents_from_cf(cf: List[int]) -> List[Fraction]:
    """Generate convergents from a continued fraction list."""
    convergents = []
    for i in range(1, len(cf) + 1):
        convergents.append(Fraction(cf[0]).limit_denominator())
        if i == 1:
            continue
        frac = Fraction(cf[i - 1])
        for j in range(i - 2, -1, -1):
            frac = cf[j] + Fraction(1, frac)
        convergents.append(frac.limit_denominator())
    # Remove duplicates while preserving order
    seen = set()
    uniq = []
    for frac in convergents:
        if frac not in seen:
            uniq.append(frac)
            seen.add(frac)
    return uniq


def postprocess_order(a: int, r: int, N: int) -> Tuple[int, int]:
    """Given order r of a mod N, attempt to compute non-trivial factors."""
    if r % 2 != 0:
        raise ValueError("Order r must be even to recover factors.")
    candidate1 = math.gcd(pow(a, r // 2, N) - 1, N)
    candidate2 = math.gcd(pow(a, r // 2, N) + 1, N)
    if candidate1 in (1, N) or candidate2 in (1, N):
        raise ValueError("Post-processing failed: only trivial factors.")
    return candidate1, candidate2


# Quick test of the helpers
assert continued_fraction_expansion(0.5)[:2] == [0, 2]
assert postprocess_order(a=2, r=4, N=15) == (3, 5)



## 3. Quantum subroutines (modular multiplication for a small N)
The heavy lifting in Shor's algorithm is the controlled modular-exponentiation block \( |x\rangle \mapsto |a^x \bmod N\rangle \). For didactic purposes we specialise to \(N = 15\) and handcraft the controlled multiplication circuits that QPE needs. This follows the standard construction from the Qiskit Textbook, giving a compact circuit that simulates quickly.


In [ ]:
def c_amod15(a: int, power: int) -> QuantumCircuit:
    """Controlled multiplication by a^(2^power) mod 15 on 4 work qubits.

    Only supports values of `a` that are coprime with 15: {2, 4, 7, 8, 11, 13}.
    The circuit definitions are the compact constructions from the Qiskit Textbook.
    """

    if math.gcd(a, 15) != 1:
        raise ValueError("a must be coprime with 15")

    # Compute a^(2^power) mod 15
    exponent = pow(a, 2**power, 15)

    # region agent log
    try:
        with open(DEBUG_LOG_PATH, "a", encoding="utf-8") as _f:
            _f.write(json.dumps({
                "sessionId": "debug-session",
                "runId": "pre-fix",
                "hypothesisId": "H1",
                "location": "shors-alg.ipynb:c_amod15",
                "message": "computed exponent",
                "data": {"a": int(a), "power": int(power), "exponent": int(exponent)},
                "timestamp": int(time.time() * 1000),
            }) + "\n")
    except Exception as _e:
        print("LOGGING FAILED (H1):", repr(_e))
    # endregion agent log

    qc = QuantumCircuit(4)
    if exponent == 1:
        # Identity multiplication: no-op on the work register (leave qc empty)
        # region agent log
        try:
            with open(DEBUG_LOG_PATH, "a", encoding="utf-8") as _f:
                _f.write(json.dumps({
                    "sessionId": "debug-session",
                    "runId": "post-fix",
                    "hypothesisId": "H5",
                    "location": "shors-alg.ipynb:c_amod15",
                    "message": "identity exponent branch taken",
                    "data": {"a": int(a), "power": int(power), "exponent": int(exponent)},
                    "timestamp": int(time.time() * 1000),
                }) + "\n")
        except Exception as _e:
            print("LOGGING FAILED (H5):", repr(_e))
        # endregion agent log
        pass
    elif exponent == 2:
        qc.swap(2, 3)
        qc.swap(1, 2)
        qc.swap(0, 1)
    elif exponent == 4:
        qc.swap(1, 3)
        qc.swap(0, 2)
    elif exponent == 7:
        qc.cx(1, 3)
        qc.cx(0, 2)
        qc.ccx(0, 2, 3)
        qc.cx(0, 2)
        qc.cx(1, 3)
        qc.swap(2, 3)
        qc.swap(1, 2)
        qc.swap(0, 1)
    elif exponent == 8:
        qc.swap(0, 1)
        qc.swap(1, 2)
        qc.swap(2, 3)
    elif exponent == 11:
        qc.swap(0, 2)
        qc.swap(1, 3)
        qc.cx(0, 2)
        qc.cx(1, 3)
        qc.ccx(0, 2, 3)
        qc.cx(0, 2)
        qc.cx(1, 3)
    elif exponent == 13:
        qc.swap(0, 1)
        qc.swap(1, 2)
        qc.swap(2, 3)
        qc.cx(1, 3)
        qc.cx(0, 2)
        qc.ccx(0, 2, 3)
        qc.cx(0, 2)
        qc.cx(1, 3)
    else:
        # region agent log
        with open(DEBUG_LOG_PATH, "a", encoding="utf-8") as _f:
            _f.write(json.dumps({
                "sessionId": "debug-session",
                "runId": "pre-fix",
                "hypothesisId": "H2",
                "location": "shors-alg.ipynb:c_amod15",
                "message": "unsupported exponent branch",
                "data": {"a": int(a), "power": int(power), "exponent": int(exponent)},
                "timestamp": int(time.time() * 1000),
            }) + "\n")
        # endregion agent log
        raise ValueError(f"Unsupported exponent {exponent} for N=15")

    # Turn into a controlled gate
    qc = qc.to_gate()
    qc.name = f"*{exponent} mod 15"
    c_gate = qc.control()
    return c_gate


## 4. Building the Shor circuit for N = 15
We now stitch together the Quantum Phase Estimation style circuit:
1. Prepare a counting register of `t` qubits in superposition via Hadamards.
2. Apply controlled modular multiplications corresponding to powers of two.
3. Apply the inverse Quantum Fourier Transform to the counting register.
4. Measure the counting register to estimate the phase \(s / r\) where \(r\) is the order.

For \(N = 15\) four work qubits hold the value register and we choose eight counting qubits to get a good resolution of the phase.


In [ ]:
def build_shor_circuit_for_15(a: int, t: int = 8) -> QuantumCircuit:
    """Create the Shor order-finding circuit for N=15 and given base a."""
    if math.gcd(a, 15) != 1:
        raise ValueError("a must be coprime with 15")

    # Counting and work registers
    counting = QuantumRegister(t, name="count")
    work = QuantumRegister(4, name="work")
    classical = ClassicalRegister(t, name="c")
    qc = QuantumCircuit(counting, work, classical)

    # 1) Initialize work register to |1>
    qc.x(work[0])

    # 2) Hadamard on counting register
    qc.h(counting)

    # 3) Controlled modular multiplication by a^(2^j)
    for j in range(t):
        # region agent log
        try:
            with open(DEBUG_LOG_PATH, "a", encoding="utf-8") as _f:
                _f.write(json.dumps({
                    "sessionId": "debug-session",
                    "runId": "pre-fix",
                    "hypothesisId": "H3",
                    "location": "shors-alg.ipynb:build_shor_circuit_for_15",
                    "message": "applying controlled multiplication",
                    "data": {"a": int(a), "j": int(j), "t": int(t)},
                    "timestamp": int(time.time() * 1000),
                }) + "\n")
        except Exception as _e:
            print("LOGGING FAILED (H3):", repr(_e))
        # endregion agent log
        qc.append(c_amod15(a, j), [counting[j]] + list(work))

    # 4) Apply inverse QFT to counting register
    qc.append(QFT(num_qubits=t, inverse=True, do_swaps=True, name="QFT†"), counting)

    # 5) Measure counting register
    qc.measure(counting, classical)
    return qc


def simulate_counts(qc: QuantumCircuit, shots: int = 2048):
    """Simulate a circuit on an available Aer backend and return the counts."""
    sim = get_simulator()
    transpiled_circ = transpile(qc, sim)
    job = sim.run(transpiled_circ, shots=shots)
    result = job.result()
    return result.get_counts()


# Smoke-test a small circuit build
_ = build_shor_circuit_for_15(a=2, t=4)



C:\Users\chpsu\AppData\Local\Temp\ipykernel_19972\1700197847.py:38: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(num_qubits=t, inverse=True, do_swaps=True, name="QFT†"), counting)


## 5. Running the demo (factoring 15)
We will:
1. Choose a random base \(a\) coprime to 15.
2. Build and simulate the circuit to obtain a histogram of measured bitstrings.
3. Convert the most frequent outcome into a phase \(\phi \approx s / r\).
4. Use continued fractions to recover a candidate \(r\).
5. Perform classical post-processing to get the non-trivial factors.


In [ ]:
def run_shor_demo_on_15(shots: int = 4096, t: int = 8, max_tries: int = 5):
    """Run Shor's order-finding demo for N=15 with a few retries if phase is ambiguous."""
    N = 15

    for attempt in range(1, max_tries + 1):
        # Pick a random a that is coprime with 15
        candidates = [a for a in range(2, N) if math.gcd(a, N) == 1]
        a = random.choice(candidates)
        print(f"Attempt {attempt}/{max_tries} - chosen base a = {a}")

        # Quick classical shortcut: if gcd already reveals a factor, exit early
        g = math.gcd(a, N)
        if g != 1:
            print(f"Found factor classically: {g}")
            return g, N // g

        circuit = build_shor_circuit_for_15(a=a, t=t)
        counts = simulate_counts(circuit, shots=shots)

        # Identify the most likely measurement outcome
        best_bitstring = max(counts, key=counts.get)
        phase_estimate = int(best_bitstring, 2) / (2 ** t)
        print(f"Most frequent outcome: {best_bitstring} -> phase ≈ {phase_estimate:.4f}")

        # region agent log
        try:
            with open(DEBUG_LOG_PATH, "a", encoding="utf-8") as _f:
                _f.write(json.dumps({
                    "sessionId": "debug-session",
                    "runId": "order-finding",
                    "hypothesisId": "H6",
                    "location": "shors-alg.ipynb:run_shor_demo_on_15",
                    "message": "phase estimate and counts",
                    "data": {
                        "attempt": attempt,
                        "a": int(a),
                        "t": int(t),
                        "best_bitstring": best_bitstring,
                        "phase_estimate": float(phase_estimate),
                        "shots": int(shots),
                    },
                    "timestamp": int(time.time() * 1000),
                }) + "\n")
        except Exception as _e:
            print("LOGGING FAILED (H6):", repr(_e))
        # endregion agent log

        # Continued fractions to guess r
        cf = continued_fraction_expansion(phase_estimate, max_depth=t)
        convergents = convergents_from_cf(cf)

        r_candidate = None
        for frac in convergents:
            if frac.denominator == 0:
                continue
            r = frac.denominator
            # Check if r is a plausible order
            if pow(a, r, N) == 1:
                r_candidate = r

                # region agent log
                try:
                    with open(DEBUG_LOG_PATH, "a", encoding="utf-8") as _f:
                        _f.write(json.dumps({
                            "sessionId": "debug-session",
                            "runId": "order-finding",
                            "hypothesisId": "H7",
                            "location": "shors-alg.ipynb:run_shor_demo_on_15",
                            "message": "found candidate order",
                            "data": {
                                "attempt": attempt,
                                "a": int(a),
                                "r": int(r),
                                "phase_estimate": float(phase_estimate),
                            },
                            "timestamp": int(time.time() * 1000),
                        }) + "\n")
                except Exception as _e:
                    print("LOGGING FAILED (H7):", repr(_e))
                # endregion agent log

                break

        if r_candidate is not None:
            print(f"Recovered order r = {r_candidate}")
            factors = postprocess_order(a=a, r=r_candidate, N=N)
            print(f"Non-trivial factors: {factors}")
            return factors

        print("Attempt failed to recover order from phase; retrying with a new base a...")

    # If we exhausted all attempts
    raise RuntimeError("Could not recover order from the measured phase after multiple attempts.")


# Execute the demo
factors = run_shor_demo_on_15()
factors


Attempt 1/5 - chosen base a = 13


C:\Users\chpsu\AppData\Local\Temp\ipykernel_19972\1700197847.py:38: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(num_qubits=t, inverse=True, do_swaps=True, name="QFT†"), counting)


Most frequent outcome: 01000000 -> phase ≈ 0.2500
Recovered order r = 4
Non-trivial factors: (3, 5)


(3, 5)

## 6. What to try next
- Inspect the circuit depth with `circuit.depth()` and the gate counts to see why scaling up quickly becomes challenging.
- Swap in different coprime bases `a` to observe how the measured phase changes.
- Increase the number of counting qubits `t` to obtain finer phase resolution (at the cost of runtime).
- For numbers other than 15 you need new modular-multiplication gadgets (or the general Shor implementation in `qiskit.algorithms.Shor` if available in your environment).
- Run on a real quantum backend if you have access; the post-processing remains the same.


## 7. Visualising the quantum circuit
The code below builds the Shor order-finding circuit for a specific choice of base `a` and counting-qubit size `t`, then draws the full quantum circuit. This is the actual quantum circuit used inside the Shor demo above (up to the choice of `a` and `t`).


In [ ]:
# Visualise an example Shor circuit for N = 15
# (this is just for inspection; the actual demo uses a random base)
example_a = 2
example_t = 4
example_circuit = build_shor_circuit_for_15(a=example_a, t=example_t)

# Try to draw using matplotlib if available, otherwise fall back to text
try:
    from qiskit.visualization import circuit_drawer
    circuit_drawer(example_circuit, output="mpl")
except Exception:
    # Text-based diagram in the notebook output
    print(example_circuit.draw())



C:\Users\chpsu\AppData\Local\Temp\ipykernel_19972\1700197847.py:38: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(num_qubits=t, inverse=True, do_swaps=True, name="QFT†"), counting)
